# OUSIA Benchmark — Base vs Fine-Tuned
Compare Qwen/Qwen3.5-4B (base) vs OUSIA adapter on key benchmarks

Benchmarks: MMLU, ARC-Challenge, HellaSwag, TruthfulQA
Run after training completes and adapter is downloaded

In [ ]:
# Install lm-evaluation-harness
!pip install -q lm-eval==0.4.4

In [ ]:
# HF token from Colab Secrets
from google.colab import userdata
import os

hf_token = userdata.get('HF_TOKEN')
os.environ['HF_TOKEN'] = hf_token

## 1. Benchmark BASE model

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

print("Loading base model...")
base_model_id = "Qwen/Qwen3.5-4B"
base_tokenizer = AutoTokenizer.from_pretrained(base_model_id, token=hf_token)
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    token=hf_token,
    device_map="cuda",
    torch_dtype=torch.bfloat16,
)
print(f"Base model loaded: {base_model.num_parameters():,} params")

In [ ]:
!lm_eval \
  --model hf \
  --model_args pretrained=Qwen/Qwen3.5-4B,token=$HF_TOKEN,dtype=bfloat16,trust_remote_code=true \
  --tasks mmlu,arc_challenge,hellaswag,truthfulqa_mc2 \
  --device cuda:0 \
  --batch_size 4 \
  --output_path /content/results/base \
  2>&1 | tee /content/results/base_log.txt

## 2. Load OUSIA adapter

In [ ]:
# Upload the adapter zip in the Files panel first, then:
!cd /content && unzip -o ousia-phase1-adapter.zip -d ./ousia-adapter/
!ls ./ousia-adapter/

In [ ]:
from peft import PeftModel

# Reload base model and apply adapter
print("Loading base model for adapter merge...")
base_model_id = "Qwen/Qwen3.5-4B"
base_tokenizer = AutoTokenizer.from_pretrained(base_model_id, token=hf_token)
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    token=hf_token,
    device_map="cuda",
    torch_dtype=torch.bfloat16,
)
print("Merging OUSIA adapter...")
peft_model_id = "/content/ousia-adapter"
ousia_model = PeftModel.from_pretrained(base_model, peft_model_id)
ousia_model = ousia_model.merge_and_unload()
print(f"OUSIA model ready: {ousia_model.num_parameters():,} params")

## 3. Benchmark OUSIA model

In [ ]:
# Save merged model to a temp dir for lm-eval
merged_path = "/content/ousia-merged"
ousia_model.save_pretrained(merged_path)
base_tokenizer.save_pretrained(merged_path)

!lm_eval \
  --model hf \
  --model_args pretrained=/content/ousia-merged,dtype=bfloat16,trust_remote_code=true \
  --tasks mmlu,arc_challenge,hellaswag,truthfulqa_mc2 \
  --device cuda:0 \
  --batch_size 4 \
  --output_path /content/results/ousia \
  2>&1 | tee /content/results/ousia_log.txt

## 4. Compare results

In [ ]:
import json
import os

def load_results(path):
    for f in os.listdir(path):
        if f.endswith('.json'):
            with open(os.path.join(path, f)) as fp:
                return json.load(fp)
    return None

base_results = load_results('/content/results/base')
ousia_results = load_results('/content/results/ousia')

tasks = ['mmlu', 'arc_challenge', 'hellaswag', 'truthfulqa_mc2']
print(f"{'Benchmark':<20} {'Base':>10} {'OUSIA':>10} {'Delta':>10}")
print('-' * 52)
for task in tasks:
    b = base_results['results'][task]['acc_norm']
    o = ousia_results['results'][task]['acc_norm']
    d = o - b
    print(f"{task:<20} {b:>10.4f} {o:>10.4f} {d:>+10.4f}")